# NB_PRF_02 · Relaciones implícitas y resumen del perfil

**Paso 2 (continuación). Solo lee bronze**: escribe en `perfil.relacion` y en `Files/perfil/<origen>/<corrida_id>/`.

Los orígenes viejos casi nunca declaran sus claves foráneas. Este notebook las busca con datos:

1. **Padres posibles**: la clave **preferida** de cada tabla, si es de una sola columna (`perfil.clave_candidata.preferida`).
2. **Hijos posibles**: columnas de otras tablas con tipo compatible y **nombre parecido** (`ClienteID` ↔ `ClienteID`, `cliente_id` ↔ tabla `clientes` con clave `id`). Con `PROBAR_TODOS = True` prueba todos los pares compatibles (más caro).
3. Por cada par mide la **inclusión**: qué proporción de los valores del hijo existe en el padre, y cuántas filas quedan huérfanas.
4. Las FK declaradas en el catálogo también se miden: una FK declarada con huérfanos es un problema de calidad del origen.

Además sugiere el **rol** de cada tabla (dimensión o hecho) y deja:

| Archivo | Qué |
|---|---|
| `perfil.json` | El del paso anterior, completado con `relaciones` y `rol_sugerido`. **Es lo que le pasás a Claude** |
| `resumen_perfil.md` | Resumen para leer: tablas, claves, PII, relaciones y un diagrama entidad-relación en Mermaid |

In [ ]:
# Parámetros
ORIGEN = "erp"
CORRIDA_ID = ""                   # vacío = la última corrida de perfil de este origen
MIN_INCLUSION = 0.98              # desde cuánto una relación no declarada se considera probable
PROBAR_TODOS = False              # True = prueba todos los pares de tipo compatible, no solo los de nombre parecido
ESQUEMA_PERFIL = "perfil"
CARPETA_SALIDA = "/lakehouse/default/Files/perfil"
PREFIJO_BRONZE = "erp_"

In [ ]:
import json, os, re, unicodedata
from datetime import datetime, timezone
from pyspark.sql import functions as F

if not CORRIDA_ID:
    CORRIDA_ID = (spark.table(f"{ESQUEMA_PERFIL}.corrida").where(F.col("origen") == ORIGEN)
                       .orderBy(F.desc("inicio")).first()["corrida_id"])
print("Corrida de perfil:", CORRIDA_ID)

filtro = (F.col("corrida_id") == CORRIDA_ID) & (F.col("origen") == ORIGEN)
tablas = {r["tabla"]: r.asDict() for r in spark.table(f"{ESQUEMA_PERFIL}.tabla").where(filtro).collect()}
columnas = [r.asDict() for r in spark.table(f"{ESQUEMA_PERFIL}.columna").where(filtro).collect()]
claves = [r.asDict() for r in spark.table(f"{ESQUEMA_PERFIL}.clave_candidata").where(filtro).collect()]
col_idx = {(c["tabla"], c["columna"]): c for c in columnas}

ENTEROS = {"tinyint", "smallint", "int", "bigint"}


def norm(s):
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]", "", s.lower())


def entidad(tabla):
    """bronze.erp_clientes -> {'clientes', 'cliente', 'client'}"""
    n = tabla.split(".")[-1].lower()
    if PREFIJO_BRONZE and n.startswith(PREFIJO_BRONZE.lower()):
        n = n[len(PREFIJO_BRONZE):]
    n = norm(n)
    formas = {n}
    if n.endswith("es"):
        formas.add(n[:-2])
    if n.endswith("s"):
        formas.add(n[:-1])
    return formas


def familia(c):
    t = c["tipo_sugerido"] or c["tipo_bronze"]
    if t in ENTEROS or re.fullmatch(r"decimal\(\d+,0\)", t or ""):
        return "entero"
    if t == "string":
        return "texto"
    return t   # date, timestamp, decimal(p,s), boolean: no se usan como clave de relación


def nombre_parecido(hijo, padre_tabla, padre_col):
    h, p = norm(hijo), norm(padre_col)
    if p not in ("id", "codigo", "cod", "code") and (h == p or (len(p) >= 4 and h.endswith(p))):
        return True
    return any(h in (e + "id", "id" + e, e + "codigo", "cod" + e, e) for e in entidad(padre_tabla))


def leer(tabla):
    df = spark.table(tabla)
    run = tablas[tabla].get("run_id_bronze")
    return df.where(F.col("_run_id") == run) if run and "_run_id" in df.columns else df

## 1. Pares a probar

In [ ]:
padres = []
for k in claves:
    cols = k["columnas"].split(",")
    if len(cols) == 1 and k["es_unica"] and k["preferida"]:
        c = col_idx.get((k["tabla"], cols[0]))
        if c and familia(c) in ("entero", "texto") and (k["tabla"], cols[0]) not in [(p["tabla"], p["columna"]) for p in padres]:
            padres.append(c)

# FK declaradas en el catálogo: esquema.tabla.columna del origen -> tabla y columna de bronze
por_origen = {(t["tabla_origen"] or "").lower(): t["tabla"] for t in tablas.values()}
declaradas = {}
for c in columnas:
    if c["fk_declarada"]:
        objeto, col_ref = c["fk_declarada"].rsplit(".", 1)
        tabla_padre = por_origen.get(objeto.lower())
        if tabla_padre:
            col_padre = next((x["columna"] for x in columnas if x["tabla"] == tabla_padre
                              and x["columna"].lower() == re.sub(r"[ ,;{}()\n\t=]+", "_", col_ref).lower()), None)
            if col_padre:
                declaradas[(c["tabla"], c["columna"])] = (tabla_padre, col_padre)

pares = {}
for (th, ch), (tp, cp) in declaradas.items():
    pares[(th, ch, tp, cp)] = "declarada"
for p in padres:
    for c in columnas:
        if (c["tabla"], c["columna"]) == (p["tabla"], p["columna"]) or familia(c) != familia(p):
            continue
        if (c["tabla"], c["columna"], p["tabla"], p["columna"]) in pares:
            continue
        if c["tabla"] == p["tabla"] and not nombre_parecido(c["columna"], p["tabla"], p["columna"]):
            continue   # autorreferencias solo por nombre (ej. jefe_id -> empleado_id no se detecta así)
        if PROBAR_TODOS or nombre_parecido(c["columna"], p["tabla"], p["columna"]):
            pares[(c["tabla"], c["columna"], p["tabla"], p["columna"])] = "inferida"
print(f"{len(padres)} columnas clave como padres posibles, {len(pares)} pares a probar")

## 2. Medir la inclusión de cada par

In [ ]:
def valores(tabla, columna):
    return leer(tabla).select(F.trim(F.col(f"`{columna}`").cast("string")).alias("v")).where("v IS NOT NULL AND v <> ''")


relaciones = []
for (th, ch, tp, cp), origen in pares.items():
    hijo = valores(th, ch).groupBy("v").count()
    padre = valores(tp, cp).distinct()
    r = hijo.agg(F.count("*").alias("distintos"), F.sum("count").alias("filas")).first()
    huerf = hijo.join(padre, "v", "left_anti").agg(F.count("*").alias("distintos"), F.sum("count").alias("filas")).first()
    distintos, huerfanos = int(r["distintos"] or 0), int(huerf["distintos"] or 0)
    inclusion = round(1 - huerfanos / distintos, 4) if distintos else 0.0
    if origen == "declarada":
        estado = "declarada_ok" if huerfanos == 0 else "declarada_con_huerfanos"
    else:
        estado = "probable" if distintos and inclusion >= MIN_INCLUSION else "descartada"
    hijo_unico = bool(col_idx[(th, ch)]["es_unica"])
    relaciones.append({
        "corrida_id": CORRIDA_ID, "origen": ORIGEN, "tabla_hijo": th, "columna_hijo": ch,
        "tabla_padre": tp, "columna_padre": cp, "tipo": origen, "estado": estado,
        "cardinalidad": "1:1" if hijo_unico else "N:1",
        "valores_hijo": distintos, "valores_huerfanos": huerfanos, "filas_huerfanas": int(huerf["filas"] or 0),
        "inclusion": inclusion,
        "ejemplos_huerfanos": None if col_idx[(th, ch)]["pii"] else json.dumps(
            [f[0] for f in hijo.join(padre, "v", "left_anti").limit(5).collect()]) if huerfanos else None,
    })
    print(f"{estado:24} {th}.{ch} -> {tp}.{cp}  inclusión {inclusion:.2%}  huérfanos {huerfanos}")

ESQUEMA_REL = """corrida_id string, origen string, tabla_hijo string, columna_hijo string, tabla_padre string,
    columna_padre string, tipo string, estado string, cardinalidad string, valores_hijo bigint,
    valores_huerfanos bigint, filas_huerfanas bigint, inclusion double, ejemplos_huerfanos string"""
campos = [c.strip().split(" ")[0] for c in ESQUEMA_REL.replace("\n", " ").split(",")]
if relaciones:
    (spark.createDataFrame([tuple(r[k] for k in campos) for r in relaciones], ESQUEMA_REL)
          .write.format("delta").mode("append").saveAsTable(f"{ESQUEMA_PERFIL}.relacion"))

## 3. Rol sugerido de cada tabla, `perfil.json` y `resumen_perfil.md`

In [ ]:
validas = [r for r in relaciones if r["estado"] != "descartada"]
MEDIDA = re.compile(r"importe|monto|precio|cantidad|total|neto|bruto|saldo|costo|qty|amount", re.I)


def rol(tabla):
    """Heurística: la decide Claude con el resto del perfil y la confirma una persona."""
    es_padre = any(r["tabla_padre"] == tabla and r["tabla_hijo"] != tabla for r in validas)
    hacia = {r["tabla_padre"] for r in validas if r["tabla_hijo"] == tabla and r["tabla_padre"] != tabla}
    cols = [c for c in columnas if c["tabla"] == tabla]
    medidas = [c["columna"] for c in cols if MEDIDA.search(c["columna"])
               and (c["tipo_sugerido"] or "").startswith(("decimal", "int", "bigint"))]
    fechas = [c["columna"] for c in cols if c["tipo_sugerido"] in ("date", "timestamp")]
    if len(hacia) >= 2 and medidas and fechas and not es_padre:
        return "hecho", f"apunta a {len(hacia)} tablas, medidas {medidas[:3]}, fecha {fechas[0]}"
    if len(hacia) >= 1 and medidas and es_padre:
        return "hecho_cabecera", "tiene medidas y otras tablas apuntan a ella (cabecera de un documento)"
    if es_padre:
        return "dimension", "otras tablas apuntan a su clave"
    if len(hacia) >= 1 and medidas:
        return "hecho", f"apunta a {sorted(hacia)} y tiene medidas {medidas[:3]}"
    return "sin_definir", "sin relaciones claras: revisar a mano"


carpeta = os.path.join(CARPETA_SALIDA, re.sub(r"[^\w\-]+", "_", ORIGEN), CORRIDA_ID)
ruta_json = os.path.join(carpeta, "perfil.json")
perfil = json.load(open(ruta_json, encoding="utf-8"))
for t in perfil["tablas"]:
    t["rol_sugerido"], t["rol_motivo"] = rol(t["tabla"])
perfil["relaciones"] = [{k: (json.loads(v) if k == "ejemplos_huerfanos" else v) for k, v in r.items()
                         if k not in ("corrida_id", "origen") and v is not None}
                        for r in validas]
perfil["relaciones_descartadas"] = len(relaciones) - len(validas)
with open(ruta_json, "w", encoding="utf-8") as f:
    json.dump(perfil, f, ensure_ascii=False, indent=1, default=str)

# Resumen legible, con diagrama
def ent_mmd(tabla):
    return re.sub(r"[^0-9A-Za-z_]", "_", tabla.split(".")[-1])


lineas = [f"# Perfil de `{ORIGEN}` · corrida {CORRIDA_ID}", "",
          "Generado por NB_PRF_01 y NB_PRF_02. Es evidencia para diseñar silver y gold, no un diseño.", "",
          "## Tablas", "", "| Tabla | Filas | Rol sugerido | Clave | PII | Watermark posible |", "|---|---|---|---|---|---|"]
for t in perfil["tablas"]:
    cl = next((",".join(k["columnas"]) for k in t["claves_candidatas"] if k.get("es_unica")), "**sin clave única**")
    pii = ", ".join(c["columna"] for c in t["columnas"] if c.get("pii")) or "—"
    wm = ", ".join(c["columna"] for c in t["columnas"] if c.get("candidata_watermark")) or "—"
    lineas.append(f"| `{t['tabla']}` | {t['filas']:,} | {t['rol_sugerido']} | {cl} | {pii} | {wm} |")

lineas += ["", "## Relaciones", "", "| Hijo | Padre | Estado | Inclusión | Filas huérfanas |", "|---|---|---|---|---|"]
for r in validas:
    lineas.append(f"| `{r['tabla_hijo']}.{r['columna_hijo']}` | `{r['tabla_padre']}.{r['columna_padre']}` "
                  f"| {r['estado']} | {r['inclusion']:.2%} | {r['filas_huerfanas']:,} |")

lineas += ["", "```mermaid", "erDiagram"]
for r in validas:
    card = "||--||" if r["cardinalidad"] == "1:1" else "||--o{"
    lineas.append(f"    {ent_mmd(r['tabla_padre'])} {card} {ent_mmd(r['tabla_hijo'])} : \"{r['columna_hijo']}\"")
lineas += ["```", "", "## Para revisar", ""]
for t in perfil["tablas"]:
    for c in t["columnas"]:
        for n in c.get("notas") or []:
            lineas.append(f"- `{t['tabla']}.{c['columna']}`: {n}")
open(os.path.join(carpeta, "resumen_perfil.md"), "w", encoding="utf-8").write("\n".join(lineas) + "\n")

print(f"{len(validas)} relaciones (de {len(relaciones)} probadas). Listo en {carpeta}")
print("Siguiente paso: pasale perfil.json a Claude con la skill disenar-medallion-desde-perfil.")

In [ ]:
notebookutils.notebook.exit(json.dumps({"corrida_id": CORRIDA_ID, "relaciones": len(validas),
                                        "carpeta": carpeta}, ensure_ascii=False))